# Language Models & Text Generation

**ANLP Session 6 — Transformers & Language Models**

---

So far in Session 6 we have focused on **BERT** — a *discriminative* model trained to understand text and produce representations for downstream tasks (sentiment, similarity, entity recognition).

This notebook introduces the other major family: **generative language models** — models trained to *produce* text.

| | Discriminative (BERT-style) | Generative (GPT-style) |
|---|---|---|
| Training objective | Masked token prediction, next-sentence prediction | Next token prediction (left-to-right) |
| Architecture | Bidirectional encoder | Unidirectional decoder |
| Primary use | Classification, NER, QA, embeddings | Text generation, chat, summarisation |
| Examples | BERT, RoBERTa, DistilBERT | GPT-2, GPT-4, DialoGPT, Llama |

We will work with two models:

1. **DialoGPT** — Microsoft's GPT-2-based conversational model. We will build a multi-turn chatbot and explore how generation parameters affect the output.
2. **BART** — Facebook's encoder-decoder model trained for summarisation. We will use it to compress long articles into concise summaries.

Both models are available on HuggingFace and run locally without any API keys.

---

## Setup

In [1]:
import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

import torch
import transformers
transformers.logging.set_verbosity_warning()

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    pipeline,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device    : {device}")
print(f"PyTorch   : {torch.__version__}")
print(f"Transformers: {transformers.__version__}")

Device    : cpu
PyTorch   : 2.10.0+cpu
Transformers: 5.2.0


---

## Part 1 — Autoregressive Text Generation

Before building a chatbot, let's understand *how* generative models produce text.

### The autoregressive process

A language model assigns a probability to the next token given all previous tokens:

```
P(token_t | token_1, token_2, ..., token_{t-1})
```

To generate a sentence, you sample tokens one at a time and feed each new token back in as additional context:

```
Prompt: "The weather today is"
Step 1: P(next | "The weather today is")         → "sunny"
Step 2: P(next | "The weather today is sunny")    → "and"
Step 3: P(next | "The weather today is sunny and") → "warm"
...
```

This is fundamentally different from BERT, which processes the entire sequence bidirectionally in a single forward pass.

---

## Part 2 — DialoGPT: A Conversational Language Model

[DialoGPT](https://arxiv.org/abs/1911.00536) (Microsoft, 2019) is GPT-2 fine-tuned on 147M Reddit conversation threads. It generates contextually appropriate conversational responses.

We will use the medium variant (345M parameters) — large enough to produce reasonable responses while still running locally.

> **Note:** DialoGPT is a demonstration of how conversational models work. It is not a production assistant — responses can be inconsistent or factually wrong. Modern assistants like ChatGPT are orders of magnitude larger and additionally trained with RLHF (Reinforcement Learning from Human Feedback).

In [2]:
# Load DialoGPT-medium (~345M parameters, ~1.3 GB)
DIALOGPT_MODEL = "microsoft/DialoGPT-medium"

dialogpt_tokenizer = AutoTokenizer.from_pretrained(DIALOGPT_MODEL)
dialogpt_model     = AutoModelForCausalLM.from_pretrained(DIALOGPT_MODEL)
dialogpt_model     = dialogpt_model.to(device)
dialogpt_model.eval()

print(f"DialoGPT loaded on {device}")
print(f"Parameters: {sum(p.numel() for p in dialogpt_model.parameters()):,}")

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


DialoGPT loaded on cpu
Parameters: 406,286,336


### 2.1 Single-Turn Generation

Let's start simple: give the model a single message and see what it returns.

DialoGPT uses a specific format: conversations are concatenated with the end-of-sequence token (`<|endoftext|>`) as a separator between turns. The model is trained to predict the next turn given all previous turns.

In [3]:
def generate_reply(user_message, history_ids=None, max_length=1000,
                   temperature=0.7, top_p=0.9):
    """
    Generate a single response from DialoGPT.

    Parameters
    ----------
    user_message : str
        The user's input text.
    history_ids : torch.Tensor or None
        Token IDs of the conversation so far (for multi-turn chat).
    max_length : int
        Maximum total length of the generated sequence.
    temperature : float
        Controls randomness. Higher = more diverse, lower = more deterministic.
    top_p : float
        Nucleus sampling: restrict choices to the top-p cumulative probability mass.

    Returns
    -------
    response : str
    new_history_ids : torch.Tensor
    """
    # Encode user input and append the end-of-sequence token
    new_input_ids = dialogpt_tokenizer.encode(
        user_message + dialogpt_tokenizer.eos_token, return_tensors="pt"
    ).to(device)

    # Concatenate with conversation history (if any)
    if history_ids is not None:
        bot_input_ids = torch.cat([history_ids, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids

    attention_mask = torch.ones(bot_input_ids.shape, dtype=torch.long, device=device)

    # Generate: sample the next tokens autoregressively
    with torch.no_grad():
        output_ids = dialogpt_model.generate(
            bot_input_ids,
            attention_mask=attention_mask,
            max_length=max_length,
            pad_token_id=dialogpt_tokenizer.eos_token_id,
            do_sample=True,           # sample from the distribution (not greedy)
            temperature=temperature,
            top_p=top_p,
        )

    # Decode only the newly generated tokens (not the prompt)
    new_tokens = output_ids[:, bot_input_ids.shape[-1]:]
    response = dialogpt_tokenizer.decode(new_tokens[0], skip_special_tokens=True)

    return response, output_ids


# Try a single message
reply, _ = generate_reply("Hello! How are you today?")
print(f"User 👤: Hello! How are you today?")
print(f"Bot 🤖 : {reply}")

User 👤: Hello! How are you today?
Bot 🤖 : I'm good! How are you?


### 2.2 Multi-Turn Chat with Conversation History

A key feature of conversational models is **context tracking** — the model's response should be influenced by the full conversation so far, not just the last message.

We achieve this by accumulating the token IDs of the entire conversation and passing them to the model on each turn. The model sees:

```
[turn 1 user tokens] [EOS] [turn 1 bot tokens] [EOS] [turn 2 user tokens] [EOS] ...
```

The cell below simulates a 5-turn conversation from a fixed script so you can see the output without interactive input.

In [4]:
# A scripted conversation — change these messages to experiment
scripted_messages = [
    "Hi! My name is Alex.",
    "What do you enjoy doing in your free time?",
    "Do you like reading books?",
    "What kind of books do you recommend?",
    "Thanks for the suggestion!",
]

history_ids = None

for user_message in scripted_messages:
    reply, history_ids = generate_reply(
        user_message, history_ids=history_ids, temperature=0.7, top_p=0.9
    )
    print(f"User 👤: {user_message}")
    print(f"Bot 🤖 : {reply}")
    print()

User 👤: Hi! My name is Alex.
Bot 🤖 : Hi Alex!

User 👤: What do you enjoy doing in your free time?
Bot 🤖 : I'm a graphic designer.

User 👤: Do you like reading books?
Bot 🤖 : I do!

User 👤: What kind of books do you recommend?
Bot 🤖 : I don't really read books, I've just been reading a lot lately. I've started to read a lot of fiction, as well.

User 👤: Thanks for the suggestion!
Bot 🤖 : Sure! I'm glad I could help.



### 2.3 Generation Parameters — Temperature and Top-p

Two parameters control how the model samples from its probability distribution over the next token:

**Temperature** scales the logits before the softmax:
- `temperature < 1.0` → distribution becomes sharper → model picks high-probability tokens → more deterministic, repetitive responses
- `temperature > 1.0` → distribution flattens → model explores lower-probability tokens → more creative but potentially incoherent
- `temperature = 1.0` → original model distribution

**Top-p (nucleus sampling)** restricts sampling to the smallest set of tokens whose cumulative probability ≥ p:
- `top_p = 0.9` → only sample from tokens covering 90% of the probability mass → prunes the long tail of unlikely tokens
- `top_p = 1.0` → sample from the full vocabulary

Let's see how the same prompt responds under different temperature settings.

In [5]:
prompt = "What do you think about artificial intelligence?"

settings = [
    (0.3, 0.9, "Low temperature  (deterministic)"),
    (0.7, 0.9, "Medium temperature (balanced)   "),
    (1.2, 0.9, "High temperature  (creative)    "),
]

print(f"Prompt: '{prompt}'")
print("=" * 70)

for temp, top_p, label in settings:
    reply, _ = generate_reply(prompt, temperature=temp, top_p=top_p)
    print(f"[{label}  temp={temp}]")
    print(f"  {reply}")
    print()

Prompt: 'What do you think about artificial intelligence?'
[Low temperature  (deterministic)  temp=0.3]
  I think it's a great idea.

[Medium temperature (balanced)     temp=0.7]
  I don't think it's a good idea.

[High temperature  (creative)      temp=1.2]
  Not gonna happen.



**What to observe:**
- Low temperature tends to produce safe, predictable responses — often grammatically correct but repetitive across multiple runs.
- High temperature produces more varied, sometimes surprising responses — but can drift off-topic or become incoherent.
- Most production systems use a temperature between 0.6 and 0.9 for a balance of creativity and coherence.

Run the cell multiple times to see the variance — with `do_sample=True`, the output changes on each run.

---

## Part 3 — BART: Sequence-to-Sequence Summarisation

[BART](https://arxiv.org/abs/1910.13461) (Facebook, 2019) is a **sequence-to-sequence** model with both an encoder and a decoder. It was pre-trained by corrupting text (masking spans, shuffling sentences, etc.) and training the model to reconstruct the original.

When fine-tuned on summarisation datasets, BART learns to compress long documents into shorter, coherent summaries — a fundamentally different task from BERT (which produces representations) or DialoGPT (which continues a conversation).

```mermaid
flowchart LR
    A["Input (long article)"] --> B["BART Encoder<br/>bidirectional<br/>reads full article once"] --> C["BART Decoder<br/>autoregressive<br/>generates summary token by token"] --> D["Output (short summary)"]
```

We use `facebook/bart-large-cnn` — fine-tuned on CNN/DailyMail news articles.

In [6]:
from transformers import AutoModelForSeq2SeqLM

# Load BART model and tokeniser directly
# (newer transformers versions removed "summarization" as a pipeline task name)
# facebook/bart-large-cnn is ~1.6 GB — downloads once and caches locally
BART_MODEL = "facebook/bart-large-cnn"

bart_tokenizer = AutoTokenizer.from_pretrained(BART_MODEL)
bart_model = AutoModelForSeq2SeqLM.from_pretrained(BART_MODEL).to(device)
bart_model.eval()

def summarize(text, max_length=130, min_length=30):
    """
    Summarise a text using BART.

    Parameters
    ----------
    text : str
        The article or document to summarise.
    max_length : int
        Maximum number of tokens in the generated summary.
    min_length : int
        Minimum number of tokens in the generated summary.
    """
    inputs = bart_tokenizer(
        text, return_tensors="pt",
        max_length=1024, truncation=True
    ).to(device)

    with torch.no_grad():
        summary_ids = bart_model.generate(
            inputs["input_ids"],
            max_length=max_length,
            min_length=min_length,
            length_penalty=2.0,   # penalise shorter sequences — encourages completeness
            num_beams=4,          # beam search: explore 4 candidates in parallel
            early_stopping=True,
        )

    return bart_tokenizer.decode(summary_ids[0], skip_special_tokens=True)


print("BART summariser loaded.")

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

BART summariser loaded.


### 3.1 Summarise a News Article

In [7]:
article = """
New York (CNN) When Liana Barrientos was 23 years old, she got married in Westchester County, New York.
A year later, she got married again in Westchester County, but to a different man and without divorcing
her first husband. Only 18 days after that marriage, she got hitched yet again. Then, Barrientos
declared "I do" five more times, sometimes only within two weeks of each other.
In 2010, she married once more, this time in the Bronx. In an application for a marriage license,
she stated it was her "first and only" marriage.
Barrientos, now 39, is facing two criminal counts of "offering a false instrument for filing in the
first degree," referring to her false statements on the 2010 marriage license application, according
to court documents.
Prosecutors said the marriages were part of an immigration scam. On Friday, she pleaded not guilty
at State Supreme Court in the Bronx, according to her attorney, Christopher Wright, who declined
to comment further.
After leaving court, Barrientos was arrested and charged with theft of service and criminal trespass
for allegedly sneaking into the New York subway through an emergency exit, said Detective Annette
Markowski, a police spokeswoman.
In total, Barrientos has been married 10 times, with nine of her marriages occurring between 1999
and 2002. All occurred either in Westchester County, Long Island, New Jersey or the Bronx. She is
believed to still be married to four men, and at one time, she was married to eight men at once,
prosecutors say.
Prosecutors said the immigration scam involved some of her husbands, who filed for permanent
residence status shortly after the marriages. Any divorces happened only after such filings were
approved. It was unclear whether any of the men will be prosecuted.
If convicted, Barrientos faces up to four years in prison. Her next court appearance is scheduled
for May 18.
"""

summary = summarize(article, max_length=130, min_length=30)

print("ORIGINAL ARTICLE")
print("-" * 60)
print(article.strip())
print()
print("SUMMARY (BART)")
print("-" * 60)
print(summary)

ORIGINAL ARTICLE
------------------------------------------------------------
New York (CNN) When Liana Barrientos was 23 years old, she got married in Westchester County, New York.
A year later, she got married again in Westchester County, but to a different man and without divorcing
her first husband. Only 18 days after that marriage, she got hitched yet again. Then, Barrientos
declared "I do" five more times, sometimes only within two weeks of each other.
In 2010, she married once more, this time in the Bronx. In an application for a marriage license,
she stated it was her "first and only" marriage.
Barrientos, now 39, is facing two criminal counts of "offering a false instrument for filing in the
first degree," referring to her false statements on the 2010 marriage license application, according
to court documents.
Prosecutors said the marriages were part of an immigration scam. On Friday, she pleaded not guilty
at State Supreme Court in the Bronx, according to her attorney, Christ

### 3.2 Controlling Summary Length

The `max_length` and `min_length` parameters control how long the generated summary can be. Let's compare a very short, a medium, and a longer summary of the same article.

In [8]:
length_settings = [
    (30, 15,  "Very short (tweet-length)"),
    (80, 30,  "Medium (2–3 sentences)   "),
    (150, 60, "Long (detailed)          "),
]

for max_len, min_len, label in length_settings:
    result = summarize(article, max_length=max_len, min_length=min_len)
    print(f"[{label}]")
    print(f"  {result}")
    print()

[Very short (tweet-length)]
  Liana Barrientos has been married 10 times, sometimes within two weeks of each other. Prosecutors say the marriages were part of an

[Medium (2–3 sentences)   ]
  Liana Barrientos has been married 10 times, sometimes within two weeks of each other. Prosecutors say the marriages were part of an immigration scam. She is facing two criminal counts of "offering a false instrument for filing in the first degree"

[Long (detailed)          ]
  Liana Barrientos has been married 10 times, sometimes within two weeks of each other. Prosecutors say the marriages were part of an immigration scam. She is facing two criminal counts of "offering a false instrument for filing in the first degree" She is accused of sneaking into the New York subway through an emergency exit.



### 3.3 Try Your Own Text

Paste any long article, report, or passage into the cell below and summarise it.

In [9]:
your_text = """
Replace this text with any long article or document you want to summarise.
The model works best with 2–10 paragraphs of news-style or formal prose.
Try a Wikipedia article, a news story, or a research abstract.
"""

if len(your_text.split()) > 30:  # only run if the text is long enough
    result = summarize(your_text, max_length=100, min_length=25)
    print("SUMMARY:")
    print(result)
else:
    print("Please replace 'your_text' with a longer passage and re-run.")

SUMMARY:
The model works best with 2–10 paragraphs of news-style or formal prose. Try a Wikipedia article, a news story, or a research abstract.


---

## Part 4 — Comparing Model Families

Now that we have worked with both BERT (discriminative) and generative models, let's explicitly compare them using the `pipeline` API.

In [11]:
# Different pipeline tasks — each powered by a different model architecture
tasks_and_inputs = [
    (
        "sentiment-analysis",
        "distilbert-base-uncased-finetuned-sst-2-english",
        "The new product launch was a massive success!",
        "BERT-family (discriminative): sentiment",
    ),
    (
        "fill-mask",
        "bert-base-uncased",
        "The capital of France is [MASK].",
        "BERT (discriminative): masked LM",
    ),
    (
        "text-generation",
        "distilgpt2",
        "The future of artificial intelligence",
        "GPT-family (generative): text generation",
    ),
]

for task, model_name, input_text, label in tasks_and_inputs:
    print(f"--- {label} ---")
    print(f"Input: {input_text}")
    try:
        pipe = pipeline(
            task, model=model_name,
            device=0 if device == "cuda" else -1
        )
        if task == "text-generation":
            result = pipe(input_text, max_new_tokens=30, do_sample=True, temperature=0.7)[0]
            print(f"Output: {result['generated_text']}")
        elif task == "fill-mask":
            results = pipe(input_text)
            top3 = [(r['token_str'].strip(), r['score']) for r in results[:3]]
            print(f"Top predictions: {top3}")
        else:
            result = pipe(input_text)[0]
            print(f"Output: {result}")
    except Exception as e:
        print(f"Error: {e}")
    print()

--- BERT-family (discriminative): sentiment ---
Input: The new product launch was a massive success!


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Output: {'label': 'POSITIVE', 'score': 0.9998584985733032}

--- BERT (discriminative): masked LM ---
Input: The capital of France is [MASK].


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Top predictions: [('paris', 0.416789710521698), ('lille', 0.07141639292240143), ('lyon', 0.06339258700609207)]

--- GPT-family (generative): text generation ---
Input: The future of artificial intelligence


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Output: The future of artificial intelligence (AI) will depend on the technical and technical capabilities of the new technologies. Although the new technologies are only available for the first time, they are



---

## Summary

| Model | Architecture | Task | Key property |
|---|---|---|---|
| DialoGPT | GPT-2 decoder | Open-domain chat | Autoregressive, left-to-right generation |
| BART | Encoder-decoder | Text summarisation | Reads full input, generates compressed output |
| BERT | Bidirectional encoder | Classification, MLM | Contextual representations, not generative |

### Generation parameters recap

| Parameter | Effect | Typical value |
|---|---|---|
| `temperature` | Sharpness of token distribution | 0.6–0.9 for chat |
| `top_p` | Probability mass to sample from | 0.85–0.95 |
| `max_length` | Maximum sequence length | Task-dependent |
| `do_sample` | Sample vs greedy/beam search | True for chat, False for summarisation |

### What's next

In **Session 7** we dive deeper into large language models: how they are trained with RLHF to follow instructions, the prompting techniques that shape their output, and the challenges (hallucination, bias, alignment) that make deploying them non-trivial.

In **Session 8** we use the OpenAI, Claude, and Ollama APIs to build production-ready LLM applications — including retrieval-augmented generation (RAG).